# Adalat-small Telephone Channel Adaptation

This one-click GPU Colab notebook performs the first serious CallWhisper-8k training experiment.

It starts from the pinned Hindi checkpoint `adalat-ai/whisper-small-hi-high-lr`, applies LoRA, and trains on the released GramVaani 100-hour training archive using a deterministic recording-group-disjoint split.

Training recipe:

- every selected GramVaani source clip is retained in its released telephone channel;
- one-third of sources receive one additional balanced 8 kHz / G.711 / GSM stress view;
- the final mix is about 75% released telephone audio and 25% extra channel stress;
- frozen Vaani, LAHAJA, and GramVaani dev benchmark rows are never used for training.

Primary gate:

> On held-out recording groups, does the adapter lower pooled telephone WER while keeping relative WER regression on the released source channel at or below 5%?

This is an internal adaptation gate, not a claim that the model beats ARTPARK. A passing adapter must still be evaluated once on the frozen Vaani and LAHAJA benchmarks.


## Before Running

1. Select **Runtime > Change runtime type > T4 GPU**.
2. Confirm these files exist in Drive:
   - `MyDrive/call-whisper/saved_datasets/GV_Train_100h.tar.gz`
   - `MyDrive/call-whisper/results/gv_train_100h_inventory/gv_train_100h_inventory.csv`
3. Run all cells.

The default `serious` profile uses 18,000 source clips, about 65 view-hours, 3,000 optimizer steps, 500 loss-evaluation views, and a 100-source paired WER audit. Checkpoints and final artifacts are saved immediately to Drive.

Persistent output:

```text
MyDrive/call-whisper/results/channel_adaptation_adalat_small_seed0/serious/
```


In [ ]:
# Mount Drive, clone from a valid working directory, and install pinned training dependencies.
import importlib
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive')
if subprocess.run(['nvidia-smi'], check=False).returncode:
    raise RuntimeError('GPU runtime required. Select Runtime > Change runtime type > T4 GPU.')

REPO_DIR = Path('/content/CallWhisper-8k')
os.chdir('/content')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run([
    'git', 'clone', '--depth', '1',
    'https://github.com/anshulLuhsna/CallWhisper-8k.git', str(REPO_DIR),
], check=True)
os.chdir(REPO_DIR)

subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg', 'libsndfile1'], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'
], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.46.3', 'peft==0.13.2', 'accelerate==0.34.2',
    'datasets==3.1.0', 'evaluate==0.4.3', 'jiwer>=3.0.4',
    'librosa>=0.10', 'soundfile>=0.12', 'tensorboard>=2.16',
    'pandas>=2.0,<3', 'tabulate>=0.9', 'tqdm>=4.66',
], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', '.'
], check=True)

SRC_DIR = REPO_DIR / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# A setup-cell rerun replaces the checkout but not modules cached by this Python process.
# Evict the package so subsequent imports always execute code from the fresh clone.
stale_modules = [
    name for name in sys.modules
    if name == 'callwhisper' or name.startswith('callwhisper.')
]
for module_name in stale_modules:
    del sys.modules[module_name]
importlib.invalidate_caches()
paired_telephony = importlib.import_module('callwhisper.datasets.paired_telephony')
if not hasattr(paired_telephony, '_conform_whisper_wav_duration'):
    raise RuntimeError('Fresh paired-audio duration conformance code was not loaded')

commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Python:', platform.python_version())
print('Repository commit:', commit)
print('Paired audio module:', paired_telephony.__file__)
print('Duration conformance: enabled')
print('GPU setup complete.')


In [ ]:
# Frozen experiment profiles. Change only RUN_PROFILE, then run all cells from the top.
import hashlib
import json
import random
from importlib.metadata import version

import numpy as np
import torch

RUN_PROFILE = 'serious'  # smoke, pilot, serious, full
SEED = 0
BASE_MODEL = 'adalat-ai/whisper-small-hi-high-lr'
BASE_MODEL_REVISION = 'e78553113fe7a483dbf82fefb2cbe4ea4b6bf901'
EVAL_FRACTION = 0.05
LANGUAGE = 'hi'
TASK = 'transcribe'
NUM_BEAMS = 1

PROFILES = {
    'smoke': {
        'max_train_sources': 256, 'max_eval_sources': 50,
        'paired_eval_sources': 20, 'max_steps': 40,
        'eval_steps': 20, 'save_steps': 20,
    },
    'pilot': {
        'max_train_sources': 4_000, 'max_eval_sources': 200,
        'paired_eval_sources': 50, 'max_steps': 800,
        'eval_steps': 100, 'save_steps': 100,
    },
    'serious': {
        'max_train_sources': 18_000, 'max_eval_sources': 500,
        'paired_eval_sources': 100, 'max_steps': 3_000,
        'eval_steps': 300, 'save_steps': 300,
    },
    'full': {
        'max_train_sources': None, 'max_eval_sources': 1_000,
        'paired_eval_sources': 200, 'max_steps': 6_000,
        'eval_steps': 500, 'save_steps': 500,
    },
}
if RUN_PROFILE not in PROFILES:
    raise ValueError(f'Unknown RUN_PROFILE: {RUN_PROFILE}')
PROFILE = PROFILES[RUN_PROFILE]

PER_DEVICE_TRAIN_BATCH_SIZE = 4
PER_DEVICE_EVAL_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 5e-5
WARMUP_RATIO = 0.10
LORA_CONFIG = {
    'r': 32,
    'lora_alpha': 64,
    'target_modules': ['q_proj', 'v_proj'],
    'lora_dropout': 0.05,
    'bias': 'none',
}

DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/call-whisper')
INVENTORY_PATH = DRIVE_PROJECT_DIR / 'results/gv_train_100h_inventory/gv_train_100h_inventory.csv'
DATASET_ARCHIVE = DRIVE_PROJECT_DIR / 'saved_datasets/GV_Train_100h.tar.gz'
OUTPUT_DIR = (
    DRIVE_PROJECT_DIR / 'results/channel_adaptation_adalat_small_seed0' / RUN_PROFILE
)
SPLIT_DIR = OUTPUT_DIR / 'splits'
CHECKPOINT_DIR = OUTPUT_DIR / 'checkpoints'
FINAL_ADAPTER_DIR = OUTPUT_DIR / 'final_adapter'
PROCESSOR_DIR = OUTPUT_DIR / 'processor'
WORK_ROOT = Path(f'/content/channel_adaptation_adalat_small_seed0_{RUN_PROFILE}')
LOCAL_DATA_ROOT = WORK_ROOT / 'dataset'
AUDIO_CACHE_DIR = WORK_ROOT / 'audio_cache'
for directory in (
    OUTPUT_DIR, SPLIT_DIR, CHECKPOINT_DIR, WORK_ROOT, LOCAL_DATA_ROOT, AUDIO_CACHE_DIR
):
    directory.mkdir(parents=True, exist_ok=True)

if not INVENTORY_PATH.exists():
    raise FileNotFoundError(f'Missing inventory: {INVENTORY_PATH}')
if not DATASET_ARCHIVE.exists():
    raise FileNotFoundError(f'Missing training archive: {DATASET_ARCHIVE}')

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

semantic_config = {
    'run_profile': RUN_PROFILE,
    'base_model': BASE_MODEL,
    'base_model_revision': BASE_MODEL_REVISION,
    'seed': SEED,
    'eval_fraction': EVAL_FRACTION,
    'profile': PROFILE,
    'training_mix': '75% released GramVaani telephone source; 25% extra balanced channel stress',
    'split_semantics': 'recording-group-disjoint; speaker IDs unavailable in released inventory',
    'conditions': [
        'original', 'bandlimit_8k', 'bandlimit_8k_g711_alaw',
        'bandlimit_8k_g711_mulaw', 'bandlimit_8k_gsm_fr',
    ],
    'batch': {
        'per_device_train': PER_DEVICE_TRAIN_BATCH_SIZE,
        'per_device_eval': PER_DEVICE_EVAL_BATCH_SIZE,
        'gradient_accumulation': GRADIENT_ACCUMULATION_STEPS,
    },
    'learning_rate': LEARNING_RATE,
    'warmup_ratio': WARMUP_RATIO,
    'lora': LORA_CONFIG,
    'attention_mask': True,
    'frozen_benchmarks_used_for_training': False,
}
config_path = OUTPUT_DIR / 'run_config.json'
if config_path.exists():
    existing = json.loads(config_path.read_text(encoding='utf-8'))
    if existing.get('semantic_config') != semantic_config:
        durable = list(CHECKPOINT_DIR.glob('checkpoint-*')) + list(FINAL_ADAPTER_DIR.glob('*'))
        if durable:
            raise RuntimeError(
                'Run configuration changed after checkpoints were saved. '
                'Use a new profile/output directory instead of mixing experiments.'
            )
config_path.write_text(
    json.dumps({'semantic_config': semantic_config, 'repo_commit': commit}, indent=2) + '\n',
    encoding='utf-8',
)
package_versions = {
    name: version(name)
    for name in ('transformers', 'peft', 'accelerate', 'datasets', 'jiwer', 'soundfile')
}
(OUTPUT_DIR / 'package_versions.json').write_text(
    json.dumps(package_versions, indent=2) + '\n', encoding='utf-8'
)
print(json.dumps(semantic_config, indent=2))
print('Persistent output:', OUTPUT_DIR)


In [ ]:
# Build deterministic, auditable train/eval manifests.
import csv

from callwhisper.datasets.channel_adaptation import (
    build_artifacts,
    build_paired_eval_views,
    deterministic_limit,
    read_csv,
    write_csv,
)

split_summary = build_artifacts(
    INVENTORY_PATH,
    SPLIT_DIR,
    eval_fraction=EVAL_FRACTION,
    seed=SEED,
    max_train_sources=PROFILE['max_train_sources'],
    max_eval_sources=PROFILE['max_eval_sources'],
)
if split_summary['recording_group_overlap'] != 0:
    raise RuntimeError('Recording-group leakage detected')

full_internal_eval = read_csv(SPLIT_DIR / 'internal_eval_source.csv')
paired_sources = deterministic_limit(
    full_internal_eval, PROFILE['paired_eval_sources'], seed=SEED + 2
)
paired_eval_views = build_paired_eval_views(paired_sources)
write_csv(SPLIT_DIR / 'paired_internal_eval_views.csv', paired_eval_views)

print(json.dumps(split_summary, indent=2))
print('Paired audit sources:', len(paired_sources))
print('Paired audit views:', len(paired_eval_views))


In [ ]:
# Copy and safely extract the GramVaani archive to local Colab disk.
import tarfile

from tqdm.auto import tqdm

LOCAL_ARCHIVE = WORK_ROOT / DATASET_ARCHIVE.name
EXTRACT_SENTINEL = LOCAL_DATA_ROOT / '.extract_complete'

def copy_with_progress(source_path: Path, destination_path: Path) -> None:
    total = source_path.stat().st_size
    with source_path.open('rb') as source_handle, destination_path.open('wb') as target_handle:
        with tqdm(total=total, unit='B', unit_scale=True, desc='Copying training archive') as bar:
            while True:
                chunk = source_handle.read(8 * 1024 * 1024)
                if not chunk:
                    break
                target_handle.write(chunk)
                bar.update(len(chunk))

if not LOCAL_ARCHIVE.exists() or LOCAL_ARCHIVE.stat().st_size != DATASET_ARCHIVE.stat().st_size:
    copy_with_progress(DATASET_ARCHIVE, LOCAL_ARCHIVE)

if not EXTRACT_SENTINEL.exists():
    print('Extracting training archive. This is local scratch data, not a repository artifact.')
    with tarfile.open(LOCAL_ARCHIVE, 'r:gz') as archive:
        archive.extractall(LOCAL_DATA_ROOT, filter='data')
    EXTRACT_SENTINEL.write_text('complete\n', encoding='utf-8')

dataset_candidates = [
    path for path in LOCAL_DATA_ROOT.rglob('GV_Train_100h')
    if (path / 'Audio').is_dir()
]
if not dataset_candidates:
    if (LOCAL_DATA_ROOT / 'Audio').is_dir():
        dataset_candidates = [LOCAL_DATA_ROOT]
    else:
        raise RuntimeError('Could not find extracted GV_Train_100h/Audio directory')
DATASET_DIR = dataset_candidates[0]

inventory_rows = read_csv(INVENTORY_PATH)
missing = [
    row['audio_path'] for row in inventory_rows
    if not (DATASET_DIR / row['audio_path']).exists()
]
if missing:
    raise FileNotFoundError(f'{len(missing)} inventory audio files are missing; first={missing[0]}')
print('Dataset directory:', DATASET_DIR)
print('Verified inventory audio files:', len(inventory_rows))


In [ ]:
# Materialize the selected 16 kHz training/evaluation views with visible progress.
from concurrent.futures import ThreadPoolExecutor

from callwhisper.datasets.paired_telephony import (
    probe_audio,
    transform_audio,
    validate_codec_support,
)

train_view_rows = read_csv(SPLIT_DIR / 'train_views.csv')
loss_eval_view_rows = read_csv(SPLIT_DIR / 'internal_eval_views.csv')
paired_eval_view_rows = read_csv(SPLIT_DIR / 'paired_internal_eval_views.csv')

codec_support = validate_codec_support()
if not all(codec_support.values()):
    raise RuntimeError(f'Colab ffmpeg lacks required codecs: {codec_support}')

def cached_audio_path(view_row: dict) -> Path:
    return AUDIO_CACHE_DIR / view_row['condition'] / f"{view_row['utterance_id']}.wav"

def ensure_view_audio(view_row: dict) -> str:
    destination = cached_audio_path(view_row)
    if not destination.exists():
        source_path = DATASET_DIR / view_row['source_audio_path']
        transform_audio(source_path, destination, view_row['condition'])
    return str(destination)

unique_views = {}
for view_row in train_view_rows + loss_eval_view_rows + paired_eval_view_rows:
    unique_views[view_row['view_id']] = view_row
views_to_cache = list(unique_views.values())
with ThreadPoolExecutor(max_workers=8) as pool:
    list(tqdm(
        pool.map(ensure_view_audio, views_to_cache),
        total=len(views_to_cache),
        desc='Building 16 kHz channel cache',
    ))

for condition in semantic_config['conditions']:
    sample = next(row for row in views_to_cache if row['condition'] == condition)
    metadata = probe_audio(cached_audio_path(sample))
    if metadata['sample_rate_hz'] != 16000 or metadata['channels'] != 1:
        raise RuntimeError(f'Invalid cached audio for {condition}: {metadata}')
    print(condition, metadata)
print('Cached views:', len(views_to_cache))


In [ ]:
# Load the pinned base model and attach a LoRA adapter.
import gc

import soundfile as sf
from peft import LoraConfig, get_peft_model
from transformers import WhisperForConditionalGeneration, WhisperProcessor

processor = WhisperProcessor.from_pretrained(
    BASE_MODEL,
    revision=BASE_MODEL_REVISION,
    language='Hindi',
    task=TASK,
)
model = WhisperForConditionalGeneration.from_pretrained(
    BASE_MODEL,
    revision=BASE_MODEL_REVISION,
)
model.config.forced_decoder_ids = None
model.config.use_cache = False
model = get_peft_model(model, LoraConfig(**LORA_CONFIG))
model.print_trainable_parameters()


In [ ]:
# Dataset and collator pass the Whisper attention mask during training.
from dataclasses import dataclass
from typing import Any

from torch.utils.data import Dataset

class ChannelViewDataset(Dataset):
    def __init__(self, rows: list[dict], processor: Any):
        self.rows = rows
        self.processor = processor

    def __len__(self) -> int:
        return len(self.rows)

    def __getitem__(self, index: int) -> dict:
        row = self.rows[index]
        audio, sample_rate = sf.read(cached_audio_path(row), dtype='float32')
        if audio.ndim != 1 or sample_rate != 16000:
            raise RuntimeError(f'Unexpected cached audio shape/rate for {row["view_id"]}')
        inputs = self.processor.feature_extractor(
            audio,
            sampling_rate=sample_rate,
            return_attention_mask=True,
        )
        return {
            'input_features': inputs.input_features[0],
            'attention_mask': inputs.attention_mask[0],
            'labels': self.processor.tokenizer(row['reference_text']).input_ids,
        }

@dataclass
class DataCollatorSpeechSeq2SeqWithAttentionMask:
    processor: Any

    def __call__(self, features: list[dict]) -> dict[str, torch.Tensor]:
        audio_features = [
            {
                'input_features': feature['input_features'],
                'attention_mask': feature['attention_mask'],
            }
            for feature in features
        ]
        batch = self.processor.feature_extractor.pad(
            audio_features, return_tensors='pt'
        )
        label_features = [{'input_ids': feature['labels']} for feature in features]
        label_batch = self.processor.tokenizer.pad(
            label_features, return_tensors='pt'
        )
        labels = label_batch['input_ids'].masked_fill(
            label_batch.attention_mask.ne(1), -100
        )
        if (
            labels.shape[1] > 0
            and (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().item()
        ):
            labels = labels[:, 1:]
        batch['labels'] = labels
        return batch

train_dataset = ChannelViewDataset(train_view_rows, processor)
loss_eval_dataset = ChannelViewDataset(loss_eval_view_rows, processor)
collator = DataCollatorSpeechSeq2SeqWithAttentionMask(processor)
smoke_batch = collator([train_dataset[0], train_dataset[1]])
print({key: tuple(value.shape) for key, value in smoke_batch.items()})
print('Train views:', len(train_dataset))
print('Loss-eval views:', len(loss_eval_dataset))


In [ ]:
# Train or resume from the latest durable Drive checkpoint.
import inspect
import time

from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
from transformers.trainer_utils import get_last_checkpoint

args_kwargs = dict(
    output_dir=str(CHECKPOINT_DIR),
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type='cosine',
    max_steps=PROFILE['max_steps'],
    weight_decay=0.01,
    max_grad_norm=1.0,
    gradient_checkpointing=True,
    fp16=True,
    logging_steps=5 if RUN_PROFILE == 'smoke' else 25,
    eval_steps=PROFILE['eval_steps'],
    save_steps=PROFILE['save_steps'],
    save_total_limit=2,
    save_strategy='steps',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    predict_with_generate=False,
    report_to=['tensorboard'],
    remove_unused_columns=False,
    label_names=['labels'],
    dataloader_num_workers=2,
)
signature = inspect.signature(Seq2SeqTrainingArguments)
args_kwargs['eval_strategy' if 'eval_strategy' in signature.parameters else 'evaluation_strategy'] = 'steps'
training_args = Seq2SeqTrainingArguments(**args_kwargs)
if hasattr(model, 'enable_input_require_grads'):
    model.enable_input_require_grads()

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_dataset,
    eval_dataset=loss_eval_dataset,
    data_collator=collator,
    tokenizer=processor.feature_extractor,
)

final_run_files = [
    FINAL_ADAPTER_DIR / 'adapter_model.safetensors',
    FINAL_ADAPTER_DIR / 'adapter_config.json',
    PROCESSOR_DIR / 'preprocessor_config.json',
    PROCESSOR_DIR / 'tokenizer_config.json',
    OUTPUT_DIR / 'train_metrics.json',
    OUTPUT_DIR / 'trainer_state.json',
]
if all(path.exists() for path in final_run_files):
    print('Final adapter already exists; skipping training:', FINAL_ADAPTER_DIR)
    train_metrics = json.loads(
        (OUTPUT_DIR / 'train_metrics.json').read_text(encoding='utf-8')
    )
else:
    last_checkpoint = get_last_checkpoint(str(CHECKPOINT_DIR))
    print('Resuming from:', last_checkpoint or 'fresh start')
    started = time.time()
    train_result = trainer.train(resume_from_checkpoint=last_checkpoint)
    train_metrics = dict(train_result.metrics)
    train_metrics['wall_time_s'] = time.time() - started
    FINAL_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
    PROCESSOR_DIR.mkdir(parents=True, exist_ok=True)
    trainer.save_model(str(FINAL_ADAPTER_DIR))
    processor.save_pretrained(str(PROCESSOR_DIR))
    (OUTPUT_DIR / 'train_metrics.json').write_text(
        json.dumps(train_metrics, indent=2) + '\n', encoding='utf-8'
    )
    trainer.state.save_to_json(str(OUTPUT_DIR / 'trainer_state.json'))
    os.sync()
print(json.dumps(train_metrics, indent=2))
print('Final adapter:', FINAL_ADAPTER_DIR)


In [ ]:
# Paired internal WER audit: same held-out recordings, same five conditions, base vs adapter.
import time

import pandas as pd
from jiwer import process_characters, process_words
from peft import PeftModel
from tqdm.auto import tqdm

from callwhisper.eval.normalizer import normalize_text

DEVICE = 'cuda'
PREDICTIONS_PATH = OUTPUT_DIR / 'paired_internal_predictions.jsonl'

def read_prediction_checkpoint(path: Path) -> dict[tuple[str, str], dict]:
    checkpoint = {}
    if path.exists():
        for line in path.read_text(encoding='utf-8').splitlines():
            if line.strip():
                row = json.loads(line)
                checkpoint[(row['model_label'], row['view_id'])] = row
    return checkpoint

def load_eval_model(model_label: str):
    base = WhisperForConditionalGeneration.from_pretrained(
        BASE_MODEL, revision=BASE_MODEL_REVISION
    )
    if model_label == 'adapted':
        base = PeftModel.from_pretrained(base, FINAL_ADAPTER_DIR).merge_and_unload()
    base.config.forced_decoder_ids = None
    base.eval()
    return base.to(DEVICE)

def transcribe(eval_model, audio_path: Path) -> tuple[str, float]:
    audio, sample_rate = sf.read(audio_path, dtype='float32')
    inputs = processor.feature_extractor(
        audio,
        sampling_rate=sample_rate,
        return_tensors='pt',
        return_attention_mask=True,
    )
    input_features = inputs.input_features.to(DEVICE)
    attention_mask = inputs.attention_mask.to(DEVICE)
    started = time.time()
    with torch.inference_mode():
        prediction_ids = eval_model.generate(
            input_features=input_features,
            attention_mask=attention_mask,
            language=LANGUAGE,
            task=TASK,
            num_beams=NUM_BEAMS,
            do_sample=False,
            max_new_tokens=225,
        )
    hypothesis = processor.batch_decode(
        prediction_ids, skip_special_tokens=True
    )[0].strip()
    return hypothesis, time.time() - started

checkpoint = read_prediction_checkpoint(PREDICTIONS_PATH)
for model_label in ('adapted', 'base'):
    pending = [
        row for row in paired_eval_view_rows
        if (model_label, row['view_id']) not in checkpoint
    ]
    print(model_label, 'pending:', len(pending), '/', len(paired_eval_view_rows))
    if pending:
        eval_model = load_eval_model(model_label)
        with PREDICTIONS_PATH.open('a', encoding='utf-8') as handle:
            for row in tqdm(pending, desc=f'Internal WER: {model_label}'):
                hypothesis, inference_s = transcribe(
                    eval_model, cached_audio_path(row)
                )
                prediction = {
                    **row,
                    'model_label': model_label,
                    'hypothesis_text': hypothesis,
                    'inference_s': inference_s,
                }
                handle.write(json.dumps(prediction, ensure_ascii=False) + '\n')
                handle.flush()
                checkpoint[(model_label, row['view_id'])] = prediction
        del eval_model
        gc.collect()
        torch.cuda.empty_cache()

predictions = list(checkpoint.values())
expected_predictions = len(paired_eval_view_rows) * 2
if len(predictions) != expected_predictions:
    raise RuntimeError(f'Prediction count mismatch: {len(predictions)} != {expected_predictions}')

def summarize(rows: list[dict], model_label: str, condition: str) -> dict:
    references = [normalize_text(row['reference_text']) for row in rows]
    hypotheses = [normalize_text(row['hypothesis_text']) for row in rows]
    word_output = process_words(references, hypotheses)
    char_output = process_characters(references, hypotheses)
    utterance_wers = [
        process_words(reference, hypothesis).wer
        for reference, hypothesis in zip(references, hypotheses)
    ]
    return {
        'model_label': model_label,
        'condition': condition,
        'files': len(rows),
        'corpus_wer': word_output.wer,
        'macro_utterance_wer': float(np.mean(utterance_wers)),
        'corpus_cer': char_output.cer,
        'substitutions': word_output.substitutions,
        'insertions': word_output.insertions,
        'deletions': word_output.deletions,
        'reference_words': word_output.hits + word_output.substitutions + word_output.deletions,
        'mean_inference_s': float(np.mean([row['inference_s'] for row in rows])),
    }

conditions = semantic_config['conditions']
telephone_conditions = [condition for condition in conditions if condition != 'original']
summary_rows = []
for model_label in ('base', 'adapted'):
    model_rows = [row for row in predictions if row['model_label'] == model_label]
    for condition in conditions:
        summary_rows.append(summarize(
            [row for row in model_rows if row['condition'] == condition],
            model_label,
            condition,
        ))
    summary_rows.append(summarize(
        [row for row in model_rows if row['condition'] in telephone_conditions],
        model_label,
        'pooled_telephone',
    ))

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUTPUT_DIR / 'paired_internal_summary.csv', index=False)
(OUTPUT_DIR / 'paired_internal_summary.md').write_text(
    summary_df.to_markdown(index=False) + '\n', encoding='utf-8'
)
display(summary_df)


In [ ]:
# Apply the explicit go/no-go rule and package the adapter for the frozen evaluation.
import tarfile

def metric(model_label: str, condition: str, column: str = 'corpus_wer') -> float:
    row = summary_df[
        (summary_df['model_label'] == model_label)
        & (summary_df['condition'] == condition)
    ]
    if len(row) != 1:
        raise RuntimeError(f'Missing metric: {model_label} {condition}')
    return float(row.iloc[0][column])

base_original = metric('base', 'original')
adapted_original = metric('adapted', 'original')
base_pooled = metric('base', 'pooled_telephone')
adapted_pooled = metric('adapted', 'pooled_telephone')
original_relative_regression = (
    (adapted_original - base_original) / base_original
    if base_original > 0 else float('inf')
)
pooled_delta = adapted_pooled - base_pooled
passed = pooled_delta < 0 and original_relative_regression <= 0.05

gate = {
    'verdict': 'pass_to_frozen_benchmarks' if passed else 'stop_and_diagnose',
    'base_original_wer': base_original,
    'adapted_original_wer': adapted_original,
    'original_relative_wer_regression': original_relative_regression,
    'base_pooled_telephone_wer': base_pooled,
    'adapted_pooled_telephone_wer': adapted_pooled,
    'adapted_minus_base_pooled_wer': pooled_delta,
    'rule': 'pooled telephone WER must improve and original relative WER regression must be <= 5%',
    'claim_scope': 'internal recording-group-held-out gate only',
    'next_step': (
        'run one frozen masked evaluation on Vaani and LAHAJA'
        if passed else
        'inspect condition errors; do not run or tune on frozen benchmarks'
    ),
}
gate_path = OUTPUT_DIR / 'adaptation_gate.json'
gate_path.write_text(json.dumps(gate, indent=2) + '\n', encoding='utf-8')
print(json.dumps(gate, indent=2))

required_paths = [
    config_path,
    OUTPUT_DIR / 'package_versions.json',
    OUTPUT_DIR / 'train_metrics.json',
    OUTPUT_DIR / 'trainer_state.json',
    OUTPUT_DIR / 'paired_internal_predictions.jsonl',
    OUTPUT_DIR / 'paired_internal_summary.csv',
    OUTPUT_DIR / 'paired_internal_summary.md',
    gate_path,
    SPLIT_DIR / 'channel_adaptation_split_summary.json',
    SPLIT_DIR / 'train_source.csv',
    SPLIT_DIR / 'internal_eval_source.csv',
    SPLIT_DIR / 'train_views.csv',
    SPLIT_DIR / 'internal_eval_views.csv',
    SPLIT_DIR / 'paired_internal_eval_views.csv',
    FINAL_ADAPTER_DIR / 'adapter_model.safetensors',
    FINAL_ADAPTER_DIR / 'adapter_config.json',
    PROCESSOR_DIR / 'preprocessor_config.json',
    PROCESSOR_DIR / 'tokenizer_config.json',
]
required_paths.extend(sorted(FINAL_ADAPTER_DIR.rglob('*')))
required_paths.extend(sorted(PROCESSOR_DIR.rglob('*')))
required_files = sorted({path for path in required_paths if path.is_file()})
missing_required = [str(path) for path in required_paths if not path.exists()]
if missing_required:
    raise FileNotFoundError(f'Missing required artifacts: {missing_required}')

hashes = {
    str(path.relative_to(OUTPUT_DIR)): hashlib.sha256(path.read_bytes()).hexdigest()
    for path in required_files
}
hash_path = OUTPUT_DIR / 'artifact_sha256.json'
hash_path.write_text(json.dumps(hashes, indent=2) + '\n', encoding='utf-8')

bundle_path = OUTPUT_DIR / f'adalat_small_channel_adapter_{RUN_PROFILE}_seed0.tar.gz'
with tarfile.open(bundle_path, 'w:gz') as archive:
    for path in required_files + [hash_path]:
        archive.add(path, arcname=path.relative_to(OUTPUT_DIR))
os.sync()

from google.colab import files
files.download(str(bundle_path))
print('COMPLETE')
print('Verdict:', gate['verdict'])
print('Bundle:', bundle_path)
print('All persistent outputs:', OUTPUT_DIR)


## Reading The Result

Open `adaptation_gate.json` first.

- `pass_to_frozen_benchmarks`: the adapter improved pooled telephone WER internally and kept relative source-channel regression within 5%. Freeze the adapter and evaluate it once on Vaani and LAHAJA.
- `stop_and_diagnose`: do not tune against the public benchmark. Inspect the per-condition internal errors, change the training recipe, and rerun under a new output directory.

Important limitations:

- GramVaani is already telephone speech. `original` means its released source channel, not pristine wideband audio.
- Speaker IDs are unavailable in the released inventory, so the split is recording-group-disjoint rather than proven speaker-disjoint.
- Extra G.711/GSM transforms are robustness stress, and may represent stacked degradation.
- Passing this internal gate does not mean the adapter beats ARTPARK. Only the untouched frozen benchmarks can support that comparison.
